# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zoro-369/flyrank-ml-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of Analysis**

One row represents the daily performance of one content page for one client.

**Tables**

- fact_content_daily_performance
- dim_content (if additional metadata is required)

**Time Window**

For development, I use **March 2026 (2026-03)**, which is a mid-panel month. This avoids using the final month of data as recommended in the assignment.

**Prediction Target**

Rank pages for content review using a proxy label indicating declining performance.

**Excluded**

I deliberately exclude any feature derived from the target or future observations because it would leak information into the model.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

### Features

- impressions
- clicks
- ctr
- avg_position
- sessions

### Label / Proxy

- trend_direction (proxy)
or
- is_declining_label

### Context

- client_hash_id
- content_hash_id
- report_date

### Excluded

- trend_pct
- any feature calculated from the target window

Reason:
These can reveal the answer directly and cause data leakage.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [16]:
!pip install -q duckdb huggingface_hub

from huggingface_hub import hf_hub_download
from google.colab import userdata
from huggingface_hub import login
import duckdb

# Login
token = userdata.get("HF_TOKEN")
login(token=token)

# Download ONLY the March 2026 partition
parquet_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=token,
)

print("Downloaded to:", parquet_path)

# Query with DuckDB
con = duckdb.connect()

march = con.execute(f"""
SELECT *
FROM read_parquet('{parquet_path}')
""").df()

print("Rows loaded:", len(march))
march.head()

Downloaded to: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 9841378


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [17]:
# Query 1: Verify grain
print("Rows:", len(march))

duplicates = march.duplicated(
    subset=[
        "client_hash_id",
        "content_hash_id",
        "report_date"
    ]
).sum()

print("Duplicate grain rows:", duplicates)

Rows: 9841378
Duplicate grain rows: 0


In [18]:
# Query 2: Row count and date span
print("Rows:", len(march))

print(
    "Date range:",
    march["report_date"].min(),
    "to",
    march["report_date"].max()
)

Rows: 9841378
Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


In [19]:
# Query 3: Availability
available = march[march["ga4_data_available"].fillna(False)]

print("Rows with GA4 available:", len(available))

Rows with GA4 available: 413966


## 4. Data limits

This warehouse snapshot is observational and cannot prove that refreshing a page causes traffic recovery.

Different clients have different history lengths, resulting in an unbalanced panel.

Some early rows contain Search Console data but not GA4 data.

The project uses a proxy label rather than a future outcome, so results should be interpreted as decision support rather than causal evidence.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.